# 03. Generate the Synthetic Postpartum Cohort

This notebook creates the Tether cohort: patients after gestational diabetes (GDM), a hypertensive disorder of pregnancy (HDP), or both, each followed for twelve months from delivery. It is step one of the synthetic pipeline; observations (blood pressure, glucose, EPDS, wearable data) are generated from this cohort in the next step.

All proportions are stated assumptions in `CONFIG` inside `tether_cohort.py`. Any engagement difference across insurance, language, or parity exists because the configuration places it there, so the cohort demonstrates the pipeline and does not constitute evidence about real patients.

In [1]:
import pandas as pd
from tether_cohort import CONFIG, generate_cohort, protocol_timeline, summarize

cohort = generate_cohort(n=400, seed=7)
print(cohort.shape)
cohort.head()

(400, 24)


,patient_id,delivery_date,follow_up_end,complication,has_hdp,has_gdm,hdp_subtype,age,bmi_prepregnancy,parity,...,discharge_sbp,discharge_dbp,on_antihypertensive,epds_baseline,epds_positive_baseline,device_bp_cuff,device_glucose_sensor,device_wearable,enrolled,engagement_logit
0,TETH-0001,2026-06-21,2027-06-21,GDM only,False,True,none,22,18.0,nulliparous,...,127,78,False,9,False,True,True,True,True,0.789
1,TETH-0002,2026-04-24,2027-04-24,GDM only,False,True,none,30,46.7,multiparous,...,117,66,False,4,False,True,True,True,False,0.750
2,TETH-0003,2026-05-04,2027-05-04,GDM only,False,True,none,33,24.2,nulliparous,...,105,68,False,6,False,True,True,True,False,1.233
3,TETH-0004,2026-06-12,2027-06-12,HDP only,True,False,gestational hypertension,36,34.7,nulliparous,...,149,98,False,3,False,True,False,False,True,0.558
4,TETH-0005,2026-04-15,2027-04-15,HDP only,True,False,gestational hypertension,30,36.4,nulliparous,...,119,100,False,7,False,True,False,True,True,1.121


## Protocol timeline

Phase 1 (safety) runs weekly from delivery through week 6, with threshold alerts only. Phase 2 (prevention) runs monthly from month 2 through month 12. Disengagement is assessed per monthly window, and retention is checked at 6 weeks, 6 months, and 12 months.

In [2]:
timeline = protocol_timeline()
timeline

,phase,period,unit,start_day,end_day,bp_readings_expected,glucose_expected,epds_due,alerting,retention_checkpoint
0,1 safety,week 0,week,0,6,14,continuous (GDM),False,threshold rules only,
1,1 safety,week 1,week,7,13,14,continuous (GDM),True,threshold rules only,
2,1 safety,week 2,week,14,20,14,continuous (GDM),False,threshold rules only,
3,1 safety,week 3,week,21,27,14,continuous (GDM),False,threshold rules only,
4,1 safety,week 4,week,28,34,14,continuous (GDM),False,threshold rules only,
5,1 safety,week 5,week,35,41,14,continuous (GDM),True,threshold rules only,6 weeks
6,2 prevention,month 2,month,42,71,8,screening prompt (GDM),False,baseline-trajectory tracking,
7,2 prevention,month 3,month,72,101,8,screening prompt (GDM),True,baseline-trajectory tracking,
8,2 prevention,month 4,month,102,131,8,none,False,baseline-trajectory tracking,
9,2 prevention,month 5,month,132,161,8,none,False,baseline-trajectory tracking,


## Cohort composition

In [3]:
summarize(cohort).set_index(['variable', 'level'])

percent
variable       level                                                      
complication   HDP only                                               56.2
               GDM only                                               33.2
               HDP + GDM                                              10.5
hdp_subtype    none                                                   33.2
               gestational hypertension                               32.8
               preeclampsia                                           17.8
               preeclampsia with severe features                      11.0
               chronic hypertension with superimposed preeclam...      5.2
insurance      Private                                                55.0
               Medicaid                                               38.0
               Uninsured/Other                                         7.0
race_ethnicity White                                                  46.2
               Hispanic                                               23.2
               Black                                                  17.0
               Asian                                                   7.0
               Other/Multiracial                                       6.5
language       English                                                85.0
               Spanish                                                10.8
               Other                                                   4.2
delivery_mode  vaginal                                                69.0
               cesarean                                               31.0
parity         multiparous                                            57.5
               nulliparous                                            42.5

In [4]:
cohort.groupby('complication').agg(
    n=('patient_id', 'size'),
    mean_sbp=('discharge_sbp', 'mean'),
    mean_dbp=('discharge_dbp', 'mean'),
    on_antihypertensive=('on_antihypertensive', 'mean'),
    epds_positive=('epds_positive_baseline', 'mean'),
    glucose_sensor=('device_glucose_sensor', 'mean'),
).round(2)

,n,mean_sbp,mean_dbp,on_antihypertensive,epds_positive,glucose_sensor
complication,,,,,,
GDM only,133,118.46,73.97,0.00,0.11,1.0
HDP + GDM,42,133.38,84.71,0.26,0.07,1.0
HDP only,225,135.37,84.56,0.31,0.10,0.0


## Enrollment and engagement propensity

`engagement_logit` is a latent score that the observation step will use to simulate dropout. The comparison below shows the configured differences; rerunning with `generate_cohort(config={'engagement_effects_on': False})` produces a cohort without them, which serves as a control for the fairness evaluation.

In [5]:
print(f"Enrolled: {cohort.enrolled.mean():.1%} (target {CONFIG['p_enrolled']:.0%})")
cohort.groupby(['insurance', 'language']).engagement_logit.agg(['size', 'mean']).round(2)

Enrolled: 61.8% (target 65%)


size  mean
insurance       language            
Medicaid        English    131  0.53
                Other        5 -0.02
                Spanish     16  0.36
Private         English    189  0.91
                Other       10  0.60
                Spanish     21  0.78
Uninsured/Other English     20  0.41
                Other        2  0.12
                Spanish      6  0.18

In [6]:
import os

os.makedirs('../data/synthetic', exist_ok=True)
cohort.to_csv('../data/synthetic/cohort.csv', index=False)
timeline.to_csv('../data/synthetic/protocol_timeline.csv', index=False)